# 04 · Multiplex Stacking and Cross-Cycle Registration
Stacks Fixed Cy2–Cy10 zarr acquisitions into a multiplexed array per position.
Each cycle is registered to the Cy1 reference using Cellpose-guided DAPI segmentation
and phase cross-correlation. Output: `multiplexed.zarr` with aligned cycles.

In [ ]:
from pathlib import Path
from collections import defaultdict
import dask.array as da
import zarr
import numpy as np
import torch
from natsort import natsorted
from tqdm.auto import tqdm
from skimage.registration import phase_cross_correlation
from skimage.filters import threshold_otsu
from scipy.ndimage import shift as ndi_shift
from cellpose import models
import napari

## 1 · Collect fixed-cycle zarr paths and stack per position
Collects all Fixed Cy2-Cy10 zarr image paths grouped by position,
stacks them along a new staining-round axis (S, T, C, Z, Y, X),
and writes each position out as `<pos>_plexed.zarr` for downstream reorganisation.

In [ ]:
root_dir   = Path('/mnt/DATA3/BPP0050')
output_dir = root_dir / 'multiplexed'
output_dir.mkdir(exist_ok=True)

# Exclude Cy1 — used as registration reference only
cy1_path = root_dir / 'BPP0050-1-Live-cell-to4i_Fixed_Cy1__2025-04-14T16_57_11-Measurement 1'
iteration_dirs = natsorted([p for p in root_dir.glob('*Fixed*/') if p != cy1_path])

# Collect per-position image paths across all staining cycles
pos_dict = defaultdict(list)
for iter_dir in tqdm(iteration_dirs, desc='Collecting'):
    zarr_root = iter_dir / 'acquisition/zarr'
    for pos_zarr in zarr_root.glob('*.zarr'):
        img_path = pos_zarr / 'images'
        if img_path.exists():
            pos_dict[pos_zarr.stem].append(img_path)

print(f'{len(pos_dict)} positions x {len(iteration_dirs)} cycles')

# Stack and save out per position
for pos_id, img_paths in tqdm(pos_dict.items(), total=len(pos_dict), desc='Stacking and saving'):
    arrays  = [da.from_zarr(str(p)) for p in natsorted(img_paths)]
    stacked = da.stack(arrays, axis=0)  # (S, T, C, Z, Y, X)

    # Normalise position name: "(2, 4)" -> "2_4"
    pos_name = pos_id.strip('()').replace(', ', '_')
    out_path = output_dir / f'{pos_name}_plexed.zarr'
    stacked.to_zarr(str(out_path), overwrite=True)

print(f'Stacked zarrs written to: {output_dir}')

## 2 · Load Cy1 registration reference
Cy1 is the DAPI/nuclear anchor for cross-cycle registration.
Load a single position for QC and shift computation.

In [ ]:
reg_zarr     = zarr.open_group(cy1_path / 'acquisition/zarr/(5, 5).zarr')
registration = reg_zarr.images[:].max(axis=2)   # (T, C, Y, X) Z-max
segmentation = reg_zarr.labels.segmentation[:]

# Stack example position for QC
example_pos  = list(pos_dict.keys())[0]
arrays       = [da.from_zarr(str(p)) for p in natsorted(pos_dict[example_pos])]
fixed_cycles = da.stack(arrays, axis=0).max(axis=3).compute()  # Z-max: (S, T, C, Y, X)
print(f'Fixed cycles shape (S, T, C, Y, X): {fixed_cycles.shape}')

viewer = napari.Viewer(title='Multiplex QC')
viewer.add_image(fixed_cycles, channel_axis=2, name='Fixed cycles')
viewer.add_image(registration, channel_axis=1, name='Cy1 reference')
viewer.add_labels(segmentation, name='Cy1 segmentation')

In [ ]:
reg_zarr = zarr.open_group(cy1_path / 'acquisition/zarr/(5, 5).zarr')
registration = reg_zarr.images[:].max(axis=2)   # (T, C, Y, X)
segmentation  = reg_zarr.labels.segmentation[:]

viewer = napari.Viewer(title='Multiplex QC')
viewer.add_image(fixed_cycles, channel_axis=2, name='Fixed cycles')
viewer.add_image(registration, channel_axis=1, name='Cy1 reference')
viewer.add_labels(segmentation, name='Cy1 segmentation')

## 4 · Cross-cycle registration via DAPI phase cross-correlation
Otsu threshold masks the reference to focus correlation on foreground nuclei.
A centre crop is used to avoid edge artefacts.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cellpose_model = models.CellposeModel(gpu=(device == 'cuda'))

def center_crop(img, size=2048):
    y, x = img.shape[-2:]
    return img[y//2 - size//2:y//2 + size//2, x//2 - size//2:x//2 + size//2]

dapi_ch  = -1
dapi_ref = registration[0, dapi_ch, ...]
thresh   = threshold_otsu(dapi_ref)
mask_ref = dapi_ref > thresh

cycle_shifts = []
for i, cycle in enumerate(tqdm(fixed_cycles, desc='Computing shifts')):
    dapi_cyc = cycle[0, dapi_ch, ...]
    shift, error, _ = phase_cross_correlation(
        center_crop(dapi_ref * mask_ref),
        center_crop(dapi_cyc * mask_ref), upsample_factor=10
    )
    cycle_shifts.append(shift)
    print(f'Cycle {i+1}: dy={shift[0]:.2f}, dx={shift[1]:.2f}, err={error:.4f}')

cycle_shifts = np.array(cycle_shifts)

## 5 · Apply shifts and visualise